In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()
RESULTS_ROOT = Path(os.environ.get("ANALYSIS_RESULTS_DIR", DATA_ROOT.parent / "analysis_code" / "results")).expanduser().resolve()


In [ ]:
# Mitochondrial genes were excluded from visualization, and the top-ranked remaining genes are shown.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Nature-style dot plots for top-ranked ODC genes in human brain scRNA-seq.

Two plots are generated from the same Top 10 ranked genes:

1. major_celltype plot
   The following excitatory-neuron subtypes are merged into:
       "Excitatory neuron"

       L5/6 NP
       L6 CT
       L5 ET
       L4/5 IT
       L6 IT CAR3
       L6 IT

   All other cell-type labels are retained unchanged.

2. detailed celltype plot
   Uses the original adata.obs["Celltype"] labels without merging.

Dot size:
    Fraction of cells expressing the gene, calculated from layers["counts"] > 0.

Dot color:
    Mean expression calculated from adata.X.

Outputs:
    - editable PDF
    - editable SVG
    - 600-dpi PNG
    - numerical CSV tables used for plotting
"""

from __future__ import annotations

from pathlib import Path
import re
import sys
import warnings

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import scipy.sparse as sp


# ============================================================
# 1. Input, output, and batch-task paths
# ============================================================

BASE_DIR = DATA_ROOT / "Figure6/brain"

GENE_RANK_DIR = BASE_DIR / "rankings"

BATCH_OUTPUT_DIR = (
    RESULTS_ROOT / "Figure6/brain/ranked_gene_expression"
)
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNA_SAMPLE_FILES = {
    "M1donor": BASE_DIR / "homo_sapiens_M1donor_rna.h5ad",
}

TARGET_CELL_TYPES = ["ODC"]


def sanitize_label(value: str) -> str:
    """
    Convert a biological label into a filesystem-safe label.

    Examples
    --------
    OPC -> L2_3_IT
    L6 IT CAR3 -> L6_IT_CAR3
    """
    value = str(value).strip()
    value = re.sub(r"[\\/\s]+", "_", value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


TARGET_CELL_TYPE_DISPLAY_NAMES = {
    "OPC": "OPC",
    "ASC": "Astrocyte",
    "ODC": "Oligodendrocyte",
    "L2/3 IT": "L2/3 IT",
}


ANALYSIS_CONFIGS = []

for target_cell_type in TARGET_CELL_TYPES:
    target_cell_type_file_label = sanitize_label(
        target_cell_type
    )

    for rna_sample, rna_h5ad_file in RNA_SAMPLE_FILES.items():
        ANALYSIS_CONFIGS.append(
            {
                "analysis_name": (
                    f"{rna_sample}__{target_cell_type_file_label}"
                ),
                "rna_sample": rna_sample,
                "target_cell_type": target_cell_type,
                "target_cell_type_file_label": (
                    target_cell_type_file_label
                ),
                "target_cell_type_display": (
                    TARGET_CELL_TYPE_DISPLAY_NAMES[
                        target_cell_type
                    ]
                ),
                "gene_rank_file": (
                    GENE_RANK_DIR
                    / (
                        "motc_homo_sapiens_"
                        f"{rna_sample}_rna_"
                        f"{target_cell_type_file_label}.csv"
                    )
                ),
                "rna_h5ad_file": rna_h5ad_file,
                "output_dir": (
                    BATCH_OUTPUT_DIR
                    / target_cell_type_file_label
                    / f"RNA_{rna_sample}"
                ),
            }
        )

# These globals are reassigned for each task so the original helper
# functions can continue writing through OUTPUT_DIR.
GENE_RANK_FILE: Path | None = None
RNA_H5AD_FILE: Path | None = None
OUTPUT_DIR = BATCH_OUTPUT_DIR
CURRENT_TARGET_CELL_TYPE = ""
CURRENT_TARGET_CELL_TYPE_FILE_LABEL = ""
CURRENT_TARGET_CELL_TYPE_DISPLAY = ""


# ============================================================
# 2. Parameters
# ============================================================

CELLTYPE_COLUMN = "Celltype"
MAJOR_CELLTYPE_COLUMN = "major_celltype"
GENE_SYMBOL_COLUMN = "gene_symbol"

TOP_N = 10

FILTER_MT_GENES = True
FILTER_RIBOSOMAL_GENES = False
MANUAL_EXCLUDE_GENES: set[str] = set()

REMOVE_ZERO_EXPRESSION_GENES = True

MIN_DOT_SIZE = 18
MAX_DOT_SIZE = 620
COLOR_CLIP_PERCENTILE = 99

SIZE_LEGEND_FRACTIONS = [
    0.10,
    0.30,
    0.50,
    0.75,
]

# Only neuronal subtypes are merged in major_celltype; OPC remains separate.
EX_SUBTYPES = [
    "L5/6 NP",
    "L6 CT",
    "L5 ET",
    "L4/5 IT",
    "L6 IT CAR3",
    "L6 IT",
]

EX_MAJOR_LABEL = "Excitatory neuron"

# Optional order for the merged major-cell-type plot.
# Any observed labels not listed here are appended automatically.
PREFERRED_MAJOR_ORDER = [
    EX_MAJOR_LABEL,
    "Inhibitory neuron",
    "Astrocyte",
    "Oligodendrocyte",
    "OPC",
    "Microglia",
    "Endothelial",
    "Pericyte",
    "Vascular",
    "Immune",
]

# Optional detailed-cell-type order.
# Listed EX subtypes appear first; all other observed labels are appended.
PREFERRED_DETAILED_ORDER = EX_SUBTYPES.copy()


# ============================================================
# 3. Matplotlib settings
# ============================================================

mpl.rcParams.update(
    {
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],

        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",

        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 11,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,

        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,

        "savefig.dpi": 600,
    }
)


NATURE_CMAP = LinearSegmentedColormap.from_list(
    "nature_expression",
    [
        "#F4F6F7",
        "#DCE8F2",
        "#AFCBE3",
        "#6CA6CD",
        "#358F9B",
        "#52A66D",
        "#A8C95F",
        "#E6C84F",
    ],
    N=256,
)


# ============================================================
# 4. General utility functions
# ============================================================

def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    if path.stat().st_size == 0:
        raise ValueError(f"File is empty: {path}")


def normalize_gene_symbol(value: object) -> str:
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "na",
        "<na>",
    }:
        return ""

    return text.upper()


def normalize_celltype(value: object) -> str:
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "na",
        "<na>",
    }:
        return ""

    return text


def mean_axis0(matrix) -> np.ndarray:
    if sp.issparse(matrix):
        return np.asarray(matrix.mean(axis=0)).ravel()

    return np.asarray(matrix).mean(axis=0)


def fraction_positive_axis0(matrix) -> np.ndarray:
    if sp.issparse(matrix):
        return np.asarray((matrix > 0).mean(axis=0)).ravel()

    return (np.asarray(matrix) > 0).mean(axis=0)


def combine_columns(matrix, column_indices: list[int]):
    sub = matrix[:, column_indices]

    if len(column_indices) == 1:
        result = sub
    elif sp.issparse(sub):
        result = sp.csr_matrix(sub.sum(axis=1))
    else:
        result = np.asarray(sub).sum(axis=1, keepdims=True)

    if sp.issparse(result):
        result = result.tocsr()

        if result.shape[1] != 1:
            result = sp.csr_matrix(
                np.asarray(result).reshape(-1, 1)
            )
    else:
        result = np.asarray(result).reshape(-1, 1)

    return result


def classify_filtered_gene(gene_symbol: str) -> str | None:
    symbol = normalize_gene_symbol(gene_symbol)

    if FILTER_MT_GENES and symbol.startswith("MT-"):
        return "mitochondrial_gene"

    if (
        FILTER_RIBOSOMAL_GENES
        and (
            symbol.startswith("RPL")
            or symbol.startswith("RPS")
        )
    ):
        return "ribosomal_gene"

    excluded = {
        normalize_gene_symbol(gene)
        for gene in MANUAL_EXCLUDE_GENES
    }

    if symbol in excluded:
        return "manual_exclusion"

    return None


def make_order(
    observed_values: list[str],
    preferred_order: list[str],
) -> list[str]:
    observed = [
        value
        for value in observed_values
        if value
    ]

    observed_set = set(observed)

    ordered = [
        value
        for value in preferred_order
        if value in observed_set
    ]

    remaining = sorted(
        observed_set - set(ordered)
    )

    return ordered + remaining


# ============================================================
# 5. Load ranked genes
# ============================================================

def load_gene_rank(path: Path) -> pd.DataFrame:
    """
    Load a ranked-gene CSV.

    Supported input formats:
      1. gene,score
      2. gene_id,score
      3. gene_id,gene_symbol,score
      4. gene,gene_symbol,score

    The current RNA attention files use ``gene,score``. In that case,
    ``gene`` is interpreted as an Ensembl gene ID and the gene symbol is
    filled later from the RNA AnnData.
    """
    rank_df = pd.read_csv(path)

    rank_df = rank_df.loc[
        :,
        ~rank_df.columns.astype(str).str.startswith("Unnamed:")
    ].copy()
    rank_df.columns = rank_df.columns.astype(str).str.strip()

    if "score" not in rank_df.columns:
        raise ValueError(
            f"{path} does not contain 'score'. "
            f"Observed columns: {rank_df.columns.tolist()}"
        )

    if "gene_id" in rank_df.columns:
        gene_id_column = "gene_id"
    elif "gene" in rank_df.columns:
        gene_id_column = "gene"
    else:
        raise ValueError(
            f"{path} must contain 'gene' or 'gene_id'. "
            f"Observed columns: {rank_df.columns.tolist()}"
        )

    rank_df = rank_df.copy()
    rank_df["score"] = pd.to_numeric(
        rank_df["score"],
        errors="coerce",
    )

    rank_df["gene_id"] = (
        rank_df[gene_id_column]
        .astype(str)
        .str.replace(r"\.\d+$", "", regex=True)
        .str.strip()
    )

    if "gene_symbol" in rank_df.columns:
        rank_df["gene_symbol"] = (
            rank_df["gene_symbol"]
            .map(normalize_gene_symbol)
        )
    else:
        rank_df["gene_symbol"] = ""

    invalid_id = (
        rank_df["gene_id"].str.lower().isin(
            {"", "nan", "none", "na", "<na>"}
        )
    )

    rank_df = rank_df[
        (~invalid_id)
        & rank_df["score"].notna()
    ].copy()

    rank_df = (
        rank_df
        .sort_values("score", ascending=False)
        .drop_duplicates(subset="gene_id", keep="first")
        .reset_index(drop=True)
    )

    rank_df["original_rank"] = np.arange(len(rank_df)) + 1

    # Filtering by symbol is deferred until symbols are recovered from
    # AnnData. Existing non-empty symbols can still be filtered now.
    rank_df["filter_reason"] = rank_df["gene_symbol"].map(
        lambda symbol: (
            classify_filtered_gene(symbol)
            if symbol
            else None
        )
    )

    filtered_out = rank_df[
        rank_df["filter_reason"].notna()
    ].copy()

    retained = rank_df[
        rank_df["filter_reason"].isna()
    ].copy()

    retained["filtered_rank"] = np.arange(len(retained)) + 1

    filtered_out.to_csv(
        OUTPUT_DIR / "filtered_ranked_genes_before_rna_matching.csv",
        index=False,
    )
    retained.to_csv(
        OUTPUT_DIR / "retained_ranked_genes_before_rna_matching.csv",
        index=False,
    )

    print("\n" + "=" * 80)
    print("Gene-rank loading")
    print("=" * 80)
    print(f"Input ranked genes: {len(rank_df):,}")
    print(f"Retained before RNA matching: {len(retained):,}")
    print(f"Input columns: {rank_df.columns.tolist()}")

    return retained


# ============================================================
# 6. Prepare AnnData gene symbols
# ============================================================

def prepare_gene_symbols(
    adata: ad.AnnData,
) -> ad.AnnData:
    if GENE_SYMBOL_COLUMN not in adata.var.columns:
        raise KeyError(
            f"{GENE_SYMBOL_COLUMN!r} is absent from adata.var."
        )

    symbols = (
        adata.var[GENE_SYMBOL_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    valid = (
        symbols.notna()
        & symbols.ne("")
        & symbols.ne("NAN")
        & symbols.ne("NONE")
        & symbols.ne("<NA>")
    )

    n_invalid = int((~valid).sum())

    if n_invalid > 0:
        print(
            f"\nRemoving {n_invalid:,} features "
            "without valid gene symbols."
        )

        adata = adata[:, valid.to_numpy()].copy()
        symbols = symbols.loc[valid].copy()

    adata.var["original_var_name"] = (
        adata.var_names.astype(str)
    )

    adata.var["gene_symbol_original_case"] = (
        adata.var[GENE_SYMBOL_COLUMN]
        .astype(str)
    )

    adata.var_names = symbols.astype(str).to_numpy()

    n_duplicates = int(
        adata.var_names.duplicated().sum()
    )

    if n_duplicates > 0:
        warnings.warn(
            f"Found {n_duplicates:,} duplicate gene-symbol "
            "features. Unique suffixes will be added."
        )

        adata.var_names_make_unique(join="-")

    adata.uns.setdefault("log1p", {})
    adata.uns["log1p"].setdefault("base", None)

    print("\n" + "=" * 80)
    print("Gene-symbol preparation")
    print("=" * 80)
    print(f"Genes retained:   {adata.n_vars:,}")
    print(f"Unique var_names: {adata.var_names.is_unique}")

    return adata


# ============================================================
# 7. Prepare detailed and major cell-type columns
# ============================================================

def prepare_celltype_columns(
    adata: ad.AnnData,
) -> ad.AnnData:
    if CELLTYPE_COLUMN not in adata.obs.columns:
        raise KeyError(
            f"{CELLTYPE_COLUMN!r} is absent from adata.obs."
        )

    # Convert explicitly to a regular string/object Series before
    # assigning a new merged label. This avoids pandas Categorical
    # errors when EX_MAJOR_LABEL is not already an existing category.
    detailed = (
        adata.obs[CELLTYPE_COLUMN]
        .astype("string")
        .map(normalize_celltype)
        .astype(object)
    )

    valid = detailed.ne("")

    removed = (
        detailed.loc[~valid]
        .value_counts(dropna=False)
        .rename_axis("celltype")
        .reset_index(name="n_cells")
    )

    removed.to_csv(
        OUTPUT_DIR / "removed_missing_celltypes.csv",
        index=False,
    )

    if not valid.all():
        print(
            f"\nRemoving {(~valid).sum():,} cells "
            "with missing cell-type labels."
        )

        adata = adata[valid.to_numpy(), :].copy()

        # Rebuild from the filtered AnnData so indices remain aligned.
        detailed = (
            adata.obs[CELLTYPE_COLUMN]
            .astype("string")
            .map(normalize_celltype)
            .astype(object)
        )

    ex_normalized = {
        normalize_celltype(value).lower()
        for value in EX_SUBTYPES
    }

    detailed_normalized = (
        detailed
        .astype(str)
        .str.strip()
        .str.lower()
    )

    ex_mask = detailed_normalized.isin(ex_normalized)

    # Keep this as an ordinary object Series during label replacement.
    major = detailed.copy().astype(object)
    major.loc[ex_mask] = EX_MAJOR_LABEL

    detailed_order = make_order(
        observed_values=detailed.astype(str).unique().tolist(),
        preferred_order=PREFERRED_DETAILED_ORDER,
    )

    major_order = make_order(
        observed_values=major.astype(str).unique().tolist(),
        preferred_order=PREFERRED_MAJOR_ORDER,
    )

    # Convert to ordered categoricals only after all replacements finish.
    adata.obs[CELLTYPE_COLUMN] = pd.Categorical(
        detailed.astype(str).to_numpy(),
        categories=detailed_order,
        ordered=True,
    )

    adata.obs[MAJOR_CELLTYPE_COLUMN] = pd.Categorical(
        major.astype(str).to_numpy(),
        categories=major_order,
        ordered=True,
    )

    mapping_table = pd.DataFrame(
        {
            "celltype": detailed.astype(str).to_numpy(),
            "major_celltype": major.astype(str).to_numpy(),
        },
        index=adata.obs_names,
    )

    mapping_summary = (
        mapping_table
        .groupby(
            ["celltype", "major_celltype"],
            observed=True,
        )
        .size()
        .reset_index(name="n_cells")
        .sort_values(
            ["major_celltype", "celltype"]
        )
    )

    mapping_summary.to_csv(
        OUTPUT_DIR / "celltype_to_major_mapping.csv",
        index=False,
    )

    print("\n" + "=" * 80)
    print("Cell-type preparation")
    print("=" * 80)

    print("\nOriginal detailed cell types:")
    print(
        adata.obs[CELLTYPE_COLUMN]
        .value_counts(sort=False)
        .to_string()
    )

    print("\nMajor cell types after EX merging:")
    print(
        adata.obs[MAJOR_CELLTYPE_COLUMN]
        .value_counts(sort=False)
        .to_string()
    )

    selected_ex_counts = mapping_summary[
        mapping_summary["major_celltype"]
        == EX_MAJOR_LABEL
    ].copy()

    print("\nEX subtypes merged into Excitatory neuron:")

    if selected_ex_counts.empty:
        print("No requested EX subtype was matched.")
    else:
        print(
            selected_ex_counts.to_string(index=False)
        )

    if selected_ex_counts.empty:
        observed_examples = (
            detailed.astype(str)
            .value_counts()
            .head(30)
            .index
            .tolist()
        )

        raise ValueError(
            "None of the requested EX_SUBTYPES were found in "
            f"adata.obs[{CELLTYPE_COLUMN!r}].\n"
            f"Requested labels: {EX_SUBTYPES}\n"
            f"Observed examples: {observed_examples}"
        )

    return adata


# ============================================================
# 8. Match ranked genes to RNA object
# ============================================================

def match_ranked_genes(
    adata: ad.AnnData,
    rank_df: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    """
    Match ranked genes to RNA features, preferring Ensembl gene ID.

    Gene symbols are recovered from adata.var[GENE_SYMBOL_COLUMN], allowing
    input rank files containing only ``gene,score``.
    """
    base_to_features: dict[str, list[str]] = {}
    gene_id_to_features: dict[str, list[str]] = {}
    feature_to_symbol: dict[str, str] = {}

    gene_id_columns = [
        column
        for column in [
            "gene_id",
            "gene_ids",
            "ensembl_id",
            "ensembl_ids",
            "original_var_name",
        ]
        if column in adata.var.columns
    ]

    for feature_index, feature in enumerate(adata.var_names):
        feature = str(feature)
        base_symbol = re.sub(r"-\d+$", "", feature)
        base_to_features.setdefault(base_symbol, []).append(feature)

        symbol = normalize_gene_symbol(
            adata.var.iloc[feature_index][GENE_SYMBOL_COLUMN]
        )
        feature_to_symbol[feature] = symbol or base_symbol

        # Original var name is frequently the Ensembl ID.
        id_candidates = [
            str(adata.var.iloc[feature_index][column])
            for column in gene_id_columns
        ]

        for candidate in id_candidates:
            candidate = re.sub(
                r"\.\d+$",
                "",
                str(candidate).strip(),
            )

            if candidate.lower() in {
                "", "nan", "none", "na", "<na>"
            }:
                continue

            gene_id_to_features.setdefault(
                candidate,
                [],
            ).append(feature)

    records = []

    for _, row in rank_df.iterrows():
        gene_id = re.sub(
            r"\.\d+$",
            "",
            str(row["gene_id"]).strip(),
        )
        input_symbol = normalize_gene_symbol(
            row.get("gene_symbol", "")
        )

        matched_features = gene_id_to_features.get(gene_id, [])
        match_method = "gene_id" if matched_features else ""

        if not matched_features and input_symbol:
            matched_features = base_to_features.get(
                input_symbol,
                [],
            )
            match_method = (
                "gene_symbol" if matched_features else ""
            )

        recovered_symbols = [
            feature_to_symbol[feature]
            for feature in matched_features
            if feature_to_symbol.get(feature, "")
        ]

        if input_symbol:
            final_symbol = input_symbol
        elif recovered_symbols:
            final_symbol = recovered_symbols[0]
        else:
            final_symbol = ""

        filter_reason = (
            classify_filtered_gene(final_symbol)
            if final_symbol
            else None
        )

        records.append(
            {
                "original_rank": int(row["original_rank"]),
                "filtered_rank_before_matching": int(
                    row["filtered_rank"]
                ),
                "gene_symbol": final_symbol,
                "score": float(row["score"]),
                "gene_id": gene_id,
                "present_in_adata": bool(matched_features),
                "match_method": match_method,
                "n_features": len(matched_features),
                "adata_features": ";".join(matched_features),
                "filter_reason": filter_reason,
            }
        )

    mapping_df = pd.DataFrame(records)
    mapping_df.to_csv(
        OUTPUT_DIR / "ranked_gene_mapping_to_rna.csv",
        index=False,
    )

    eligible = mapping_df[
        mapping_df["present_in_adata"]
        & mapping_df["gene_symbol"].ne("")
        & mapping_df["filter_reason"].isna()
    ].copy()

    eligible = (
        eligible
        .sort_values("original_rank")
        .drop_duplicates("gene_symbol", keep="first")
        .reset_index(drop=True)
    )
    eligible["filtered_rank"] = np.arange(len(eligible)) + 1

    selected = eligible.head(TOP_N).copy()

    if len(selected) < TOP_N:
        warnings.warn(
            f"Only {len(selected)} eligible ranked genes were found "
            f"in RNA; requested TOP_N={TOP_N}."
        )

    if selected.empty:
        raise ValueError(
            "None of the ranked genes could be matched to RNA."
        )

    selected["plot_rank"] = np.arange(len(selected)) + 1
    selected.to_csv(
        OUTPUT_DIR / "genes_used_in_both_dotplots.csv",
        index=False,
    )

    eligible.to_csv(
        OUTPUT_DIR / "retained_ranked_genes_after_rna_matching.csv",
        index=False,
    )

    print("\n" + "=" * 80)
    print("Genes used in both dot plots")
    print("=" * 80)
    print(
        selected[
            [
                "plot_rank",
                "original_rank",
                "filtered_rank",
                "gene_symbol",
                "score",
                "gene_id",
                "match_method",
            ]
        ].to_string(index=False)
    )

    return selected, base_to_features


# ============================================================
# 9. Build compact selected-gene AnnData
# ============================================================

def build_selected_gene_adata(
    adata: ad.AnnData,
    selected: pd.DataFrame,
    base_to_features: dict[str, list[str]],
) -> ad.AnnData:
    gene_order = (
        selected["gene_symbol"].tolist()
    )

    expression_columns = []
    count_columns = []

    if "counts" in adata.layers:
        count_source = adata.layers["counts"]
    else:
        warnings.warn(
            "layers['counts'] was not found. "
            "Detection fractions will use adata.X."
        )
        count_source = adata.X

    for gene_symbol in gene_order:
        feature_names = base_to_features[gene_symbol]

        feature_indices = [
            int(adata.var_names.get_loc(feature_name))
            for feature_name in feature_names
        ]

        expression_columns.append(
            combine_columns(
                adata.X,
                feature_indices,
            )
        )

        count_columns.append(
            combine_columns(
                count_source,
                feature_indices,
            )
        )

    if any(
        sp.issparse(column)
        for column in expression_columns
    ):
        expression_matrix = sp.hstack(
            [
                column
                if sp.issparse(column)
                else sp.csr_matrix(column)
                for column in expression_columns
            ],
            format="csr",
        )
    else:
        expression_matrix = np.hstack(
            expression_columns
        )

    if any(
        sp.issparse(column)
        for column in count_columns
    ):
        count_matrix = sp.hstack(
            [
                column
                if sp.issparse(column)
                else sp.csr_matrix(column)
                for column in count_columns
            ],
            format="csr",
        )
    else:
        count_matrix = np.hstack(count_columns)

    gene_var = (
        selected
        .set_index("gene_symbol")
        .loc[
            gene_order,
            [
                "plot_rank",
                "original_rank",
                "filtered_rank",
                "score",
                "gene_id",
            ],
        ]
        .copy()
    )

    selected_adata = ad.AnnData(
        X=expression_matrix,
        obs=adata.obs.copy(),
        var=gene_var,
    )

    selected_adata.layers["counts"] = count_matrix
    selected_adata.uns["log1p"] = {"base": None}

    return selected_adata


# ============================================================
# 10. Calculate expression summary for any grouping column
# ============================================================

def calculate_expression_summary(
    selected_adata: ad.AnnData,
    group_column: str,
    output_name: str,
) -> pd.DataFrame:
    if group_column not in selected_adata.obs.columns:
        raise KeyError(
            f"{group_column!r} is absent from selected_adata.obs."
        )

    series = selected_adata.obs[group_column]

    if not isinstance(series.dtype, pd.CategoricalDtype):
        categories = sorted(
            series.astype(str).unique().tolist()
        )
    else:
        categories = series.cat.categories.tolist()

    records = []

    for cell_type in categories:
        cell_mask = (
            selected_adata.obs[group_column]
            .astype(str)
            .eq(cell_type)
            .to_numpy()
        )

        n_cells = int(cell_mask.sum())

        if n_cells == 0:
            continue

        expression_sub = selected_adata.X[
            cell_mask,
            :,
        ]

        counts_sub = selected_adata.layers["counts"][
            cell_mask,
            :,
        ]

        mean_expression = mean_axis0(
            expression_sub
        )

        fraction_expressed = fraction_positive_axis0(
            counts_sub
        )

        for gene_index, gene_symbol in enumerate(
            selected_adata.var_names
        ):
            records.append(
                {
                    "grouping_column": group_column,
                    "cell_type": str(cell_type),
                    "gene_symbol": str(gene_symbol),
                    "plot_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "plot_rank",
                        ]
                    ),
                    "original_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "original_rank",
                        ]
                    ),
                    "filtered_rank": int(
                        selected_adata.var.loc[
                            gene_symbol,
                            "filtered_rank",
                        ]
                    ),
                    "score": float(
                        selected_adata.var.loc[
                            gene_symbol,
                            "score",
                        ]
                    ),
                    "gene_id": selected_adata.var.loc[
                        gene_symbol,
                        "gene_id",
                    ],
                    "n_cells": n_cells,
                    "mean_expression": float(
                        mean_expression[gene_index]
                    ),
                    "fraction_expressed": float(
                        fraction_expressed[gene_index]
                    ),
                }
            )

    summary = pd.DataFrame(records)

    summary.to_csv(
        OUTPUT_DIR / output_name,
        index=False,
    )

    return summary


# ============================================================
# 11. Nature-style dot plot
# ============================================================

def plot_dotplot(
    summary: pd.DataFrame,
    selected: pd.DataFrame,
    group_order: list[str],
    title: str,
    output_prefix: str,
) -> None:
    gene_order = (
        selected
        .sort_values("plot_rank")["gene_symbol"]
        .tolist()
    )

    observed_groups = set(
        summary["cell_type"].astype(str)
    )

    cell_order = [
        group
        for group in group_order
        if group in observed_groups
    ]

    # Defensive append in case an observed group was absent from group_order.
    cell_order.extend(
        sorted(observed_groups - set(cell_order))
    )

    if REMOVE_ZERO_EXPRESSION_GENES:
        expressed_genes = set(
            summary.loc[
                summary["fraction_expressed"] > 0,
                "gene_symbol",
            ]
        )

        gene_order = [
            gene
            for gene in gene_order
            if gene in expressed_genes
        ]

    if not gene_order:
        raise ValueError(
            f"No genes remain for plot: {output_prefix}"
        )

    plotting = summary[
        summary["gene_symbol"].isin(gene_order)
        & summary["cell_type"].isin(cell_order)
    ].copy()

    gene_to_y = {
        gene: index
        for index, gene in enumerate(gene_order)
    }

    cell_to_x = {
        cell_type: index
        for index, cell_type in enumerate(cell_order)
    }

    plotting["x"] = plotting["cell_type"].map(cell_to_x)
    plotting["y"] = plotting["gene_symbol"].map(gene_to_y)

    plotting["dot_size"] = (
        MIN_DOT_SIZE
        + plotting["fraction_expressed"]
        * (MAX_DOT_SIZE - MIN_DOT_SIZE)
    )

    color_max = np.nanpercentile(
        plotting["mean_expression"],
        COLOR_CLIP_PERCENTILE,
    )

    if not np.isfinite(color_max) or color_max <= 0:
        color_max = 1.0

    figure_width = max(
        8.0,
        0.72 * len(cell_order) + 3.5,
    )

    figure_height = max(
        4.8,
        0.48 * len(gene_order) + 1.5,
    )

    fig = plt.figure(
        figsize=(figure_width, figure_height)
    )

    # Leave more horizontal room when detailed groups are numerous.
    main_left = 0.17
    main_width = 0.66

    ax = fig.add_axes(
        [
            main_left,
            0.22,
            main_width,
            0.66,
        ]
    )

    scatter = ax.scatter(
        plotting["x"],
        plotting["y"],
        s=plotting["dot_size"],
        c=plotting["mean_expression"],
        cmap=NATURE_CMAP,
        vmin=0,
        vmax=color_max,
        edgecolors="#4C4C4C",
        linewidths=0.55,
        zorder=3,
    )

    ax.set_xticks(range(len(cell_order)))
    ax.set_xticklabels(
        cell_order,
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticks(range(len(gene_order)))

    score_lookup = (
        selected
        .set_index("gene_symbol")["score"]
        .to_dict()
    )

    rank_lookup = (
        selected
        .set_index("gene_symbol")["plot_rank"]
        .to_dict()
    )

    y_labels = [
        f"{rank_lookup[gene]}. {gene} ({score_lookup[gene]:.3f})"
        for gene in gene_order
    ]

    ax.set_yticklabels(y_labels)
    ax.invert_yaxis()

    ax.set_xlabel("")
    ax.set_ylabel(
        "Ranked gene (attention score)",
        labelpad=8,
    )
    ax.set_title(title, pad=12)

    ax.set_xlim(-0.65, len(cell_order) - 0.35)
    ax.set_ylim(len(gene_order) - 0.35, -0.65)

    ax.grid(
        axis="both",
        color="#E7E7E7",
        linewidth=0.55,
        linestyle="-",
        zorder=0,
    )
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    colorbar_ax = fig.add_axes(
        [0.855, 0.25, 0.018, 0.43]
    )

    colorbar = fig.colorbar(
        scatter,
        cax=colorbar_ax,
    )
    colorbar.set_label(
        "Mean expression in adata.X",
        rotation=90,
        labelpad=8,
    )
    colorbar.outline.set_linewidth(0.8)

    legend_ax = fig.add_axes(
        [0.84, 0.70, 0.15, 0.20]
    )
    legend_ax.axis("off")
    legend_ax.set_xlim(0, 1)
    legend_ax.set_ylim(0, 1)

    legend_ax.text(
        0.02,
        0.97,
        "Cells expressing",
        fontsize=8,
        ha="left",
        va="top",
    )

    y_positions = np.linspace(
        0.76,
        0.14,
        len(SIZE_LEGEND_FRACTIONS),
    )

    for fraction, y_position in zip(
        SIZE_LEGEND_FRACTIONS,
        y_positions,
    ):
        dot_size = (
            MIN_DOT_SIZE
            + fraction
            * (MAX_DOT_SIZE - MIN_DOT_SIZE)
        )

        legend_ax.scatter(
            [0.24],
            [y_position],
            s=dot_size,
            facecolor="#D1D4D5",
            edgecolor="#4C4C4C",
            linewidth=0.55,
        )

        legend_ax.text(
            0.50,
            y_position,
            f"{fraction * 100:.0f}%",
            fontsize=8,
            va="center",
            ha="left",
        )

    output_base = OUTPUT_DIR / output_prefix

    fig.savefig(
        output_base.with_suffix(".pdf"),
        bbox_inches="tight",
    )
    fig.savefig(
        output_base.with_suffix(".svg"),
        bbox_inches="tight",
    )
    fig.savefig(
        output_base.with_suffix(".png"),
        dpi=600,
        bbox_inches="tight",
    )

    plt.close(fig)

    print("\nSaved:")
    print(output_base.with_suffix(".pdf"))
    print(output_base.with_suffix(".svg"))
    print(output_base.with_suffix(".png"))


# ============================================================
# 12. Main
# ============================================================

def run_one_analysis(config: dict) -> dict:
    global GENE_RANK_FILE
    global RNA_H5AD_FILE
    global OUTPUT_DIR
    global CURRENT_TARGET_CELL_TYPE
    global CURRENT_TARGET_CELL_TYPE_FILE_LABEL
    global CURRENT_TARGET_CELL_TYPE_DISPLAY

    GENE_RANK_FILE = Path(config["gene_rank_file"])
    RNA_H5AD_FILE = Path(config["rna_h5ad_file"])
    OUTPUT_DIR = Path(config["output_dir"])
    CURRENT_TARGET_CELL_TYPE = str(
        config["target_cell_type"]
    )
    CURRENT_TARGET_CELL_TYPE_FILE_LABEL = str(
        config["target_cell_type_file_label"]
    )
    CURRENT_TARGET_CELL_TYPE_DISPLAY = str(
        config["target_cell_type_display"]
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 88)
    print(f"Analysis: {config['analysis_name']}")
    print(f"Target cell type: {CURRENT_TARGET_CELL_TYPE}")
    print(f"RNA sample: {config['rna_sample']}")
    print(f"Gene rank: {GENE_RANK_FILE}")
    print(f"RNA h5ad: {RNA_H5AD_FILE}")
    print(f"Output: {OUTPUT_DIR}")
    print("#" * 88)

    require_file(GENE_RANK_FILE)
    require_file(RNA_H5AD_FILE)

    rank_df = load_gene_rank(GENE_RANK_FILE)

    print("\nLoading human RNA AnnData")
    adata = ad.read_h5ad(RNA_H5AD_FILE)
    print(adata)

    adata = prepare_gene_symbols(adata)
    adata = prepare_celltype_columns(adata)

    selected, base_to_features = match_ranked_genes(
        adata=adata,
        rank_df=rank_df,
    )

    selected_adata = build_selected_gene_adata(
        adata=adata,
        selected=selected,
        base_to_features=base_to_features,
    )

    major_summary = calculate_expression_summary(
        selected_adata=selected_adata,
        group_column=MAJOR_CELLTYPE_COLUMN,
        output_name=(
            "top10_ranked_gene_expression_by_major_celltype.csv"
        ),
    )

    major_order = (
        selected_adata.obs[MAJOR_CELLTYPE_COLUMN]
        .cat.categories.tolist()
    )

    plot_dotplot(
        summary=major_summary,
        selected=selected,
        group_order=major_order,
        title=(
            f"Top 10 {CURRENT_TARGET_CELL_TYPE_DISPLAY} "
            "attention-ranked genes across EX-merged major cell types"
        ),
        output_prefix=(
            f"top10_{CURRENT_TARGET_CELL_TYPE_FILE_LABEL}_ranked_genes_"
            "major_celltype_dotplot_nature"
        ),
    )

    detailed_summary = calculate_expression_summary(
        selected_adata=selected_adata,
        group_column=CELLTYPE_COLUMN,
        output_name=(
            "top10_ranked_gene_expression_by_detailed_celltype.csv"
        ),
    )

    detailed_order = (
        selected_adata.obs[CELLTYPE_COLUMN]
        .cat.categories.tolist()
    )

    plot_dotplot(
        summary=detailed_summary,
        selected=selected,
        group_order=detailed_order,
        title=(
            f"Top 10 {CURRENT_TARGET_CELL_TYPE_DISPLAY} "
            "attention-ranked genes across detailed cell types"
        ),
        output_prefix=(
            f"top10_{CURRENT_TARGET_CELL_TYPE_FILE_LABEL}_ranked_genes_"
            "detailed_celltype_dotplot_nature"
        ),
    )

    return {
        "analysis_name": config["analysis_name"],
        "rna_sample": config["rna_sample"],
        "target_cell_type": CURRENT_TARGET_CELL_TYPE,
        "gene_rank_file": str(GENE_RANK_FILE),
        "rna_h5ad_file": str(RNA_H5AD_FILE),
        "output_dir": str(OUTPUT_DIR),
        "n_ranked_genes_loaded": len(rank_df),
        "n_genes_plotted": len(selected),
        "status": "SUCCESS",
        "error": "",
    }


def main() -> None:
    print("=" * 88)
    print("Batch RNA ranked-gene expression dot plots")
    print("=" * 88)
    print(f"Number of tasks: {len(ANALYSIS_CONFIGS)}")
    print(f"Batch output: {BATCH_OUTPUT_DIR}")

    results: list[dict] = []

    for task_index, config in enumerate(
        ANALYSIS_CONFIGS,
        start=1,
    ):
        print(
            f"\n[{task_index}/{len(ANALYSIS_CONFIGS)}] "
            f"{config['analysis_name']}"
        )

        try:
            result = run_one_analysis(config)
            results.append(result)
            print("Status: SUCCESS")

        except Exception as exc:
            print(
                f"Status: FAILED | {type(exc).__name__}: {exc}",
                file=sys.stderr,
            )

            results.append(
                {
                    "analysis_name": config["analysis_name"],
                    "rna_sample": config["rna_sample"],
                    "target_cell_type": config["target_cell_type"],
                    "gene_rank_file": str(config["gene_rank_file"]),
                    "rna_h5ad_file": str(config["rna_h5ad_file"]),
                    "output_dir": str(config["output_dir"]),
                    "n_ranked_genes_loaded": np.nan,
                    "n_genes_plotted": np.nan,
                    "status": "FAILED",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    summary = pd.DataFrame(results)

    print("\n" + "=" * 88)
    print("Batch completed")
    print("=" * 88)
    print(
        summary[
            [
                "analysis_name",
                "status",
                "n_genes_plotted",
                "output_dir",
            ]
        ].to_string(index=False)
    )

    failed = summary["status"].eq("FAILED").sum()
    if failed:
        print(
            f"Warning: {failed}/{len(summary)} tasks failed. "
            "See the logged errors.",
            file=sys.stderr,
        )
        raise RuntimeError(f"{failed}/{len(summary)} ranked-gene expression tasks failed")


if __name__ == "__main__":
    main()
